In [5]:
pip install anthropic


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
pip install ipykernel

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ============================================================
# PHASE 1: Simple chatbot with conversation history
# ============================================================
# What this file teaches:
#   - How to call the Anthropic API from Python
#   - What a "message" looks like (role + content)
#   - Why we send the FULL history every time (Claude has no memory)
#   - What a system prompt is and why it matters
# ============================================================

# Import the Anthropic library we installed with: pip install anthropic
import anthropic

# ── STEP 1: Create the client ────────────────────────────────
# This creates a connection to the Anthropic API.
# It automatically reads your API key from an environment variable
# called ANTHROPIC_API_KEY. Set it in your terminal like this:
#   Mac/Linux:  export ANTHROPIC_API_KEY="sk-ant-..."
#   Windows:    set ANTHROPIC_API_KEY=sk-ant-...
# Never hardcode your API key directly in the code!
client = anthropic.Anthropic()


# ── STEP 2: Define a system prompt ───────────────────────────
# The system prompt is a special instruction that sets the
# "personality" and rules for Claude BEFORE the conversation starts.
# The user never sees this — it's your backstage instruction.
# Think of it like a job description you give Claude before the chat begins.
SYSTEM_PROMPT = """
You are a helpful assistant who explains things clearly 
and concisely. When answering, always check if the user 
seems confused and offer to explain further if needed.
"""


# ── STEP 3: Create the history list ──────────────────────────
# This is the most important concept in Phase 1.
# Claude has NO memory between API calls — every call starts fresh.
# So WE have to remember the conversation and send it every time.
#
# Each message in the list is a dictionary with two keys:
#   "role"    → who sent this message: "user" or "assistant"
#   "content" → the actual text of the message
#
# We start with an empty list. Messages get added as the chat goes on.
conversation_history = []


# ── STEP 4: Define a function to chat ────────────────────────
# This function handles one full round-trip:
#   1. Add the user's message to history
#   2. Send the entire history to Claude
#   3. Get Claude's reply
#   4. Add Claude's reply to history
#   5. Return the reply text so we can print it
def chat(user_message):
    """
    Send a user message to Claude and get a reply.
    Automatically maintains the conversation history.
    
    Args:
        user_message (str): What the user typed
    
    Returns:
        str: Claude's reply text
    """

    # Add the user's new message to our history list.
    # We append a dict with role="user" and their message as content.
    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    # Call the Anthropic API, sending the FULL history every time.
    # This is how Claude "remembers" — it re-reads the whole conversation.
    #
    # Parameters explained:
    #   model       → which version of Claude to use
    #   max_tokens  → maximum length of the reply (1 token ≈ 0.75 words)
    #   system      → the system prompt (Claude's instructions)
    #   messages    → the full conversation history so far
    response = client.messages.create(
        model="claude-sonnet-4-5",       # Use Claude Sonnet (fast + smart)
        max_tokens=1024,                  # Allow up to ~750 words in reply
        system=SYSTEM_PROMPT,             # Our backstage instructions
        messages=conversation_history     # The FULL history — every message
    )

    # Extract the reply text from the response object.
    # response.content is a list of "blocks" (usually just one text block).
    # We grab the first block's text.
    assistant_reply = response.content[0].text

    # Add Claude's reply to our history too.
    # Now BOTH sides of the conversation are recorded.
    # Next time we call this function, Claude will see this reply
    # and know what it already said.
    conversation_history.append({
        "role": "assistant",
        "content": assistant_reply
    })

    # Return the reply text so we can print it
    return assistant_reply


# ── STEP 5: The main loop ─────────────────────────────────────
# This is the actual chatbot interface.
# It keeps asking for input until the user types "quit".
def main():
    print("=" * 50)
    print("Phase 1 Chatbot — type 'quit' to exit")
    print("=" * 50)

    # Keep looping forever until the user quits
    while True:

        # Get the user's input from the terminal
        user_input = input("\nYou: ").strip()

        # .strip() removes any accidental spaces or newlines
        # around the user's input

        # If the user typed nothing, ask again
        if not user_input:
            print("(Please type something!)")
            continue  # 'continue' skips to the next loop iteration

        # If the user wants to quit, break out of the loop
        if user_input.lower() == "quit":
            print("Goodbye!")
            break  # 'break' exits the while loop entirely

        # ── BONUS: Show what's in the history ──
        # Uncomment the lines below if you want to SEE the history
        # growing with each message — great for learning!
        # print(f"\n[DEBUG] History has {len(conversation_history)} messages so far")

        # Send the message to Claude and get a reply
        reply = chat(user_input)

        # Print Claude's reply
        print(f"\nClaude: {reply}")


# ── Standard Python entry point ───────────────────────────────
# This means: only run main() if this file is run directly
# (not if it's imported as a module by another file).
# It's a Python best practice you'll see everywhere.
if __name__ == "__main__":
    main()